In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, spark_partition_id, lit
import pandas as pd

In [2]:
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

df = (spark
      .read
      .option("header", "true")
      .csv("/home/iceberg/data/events.csv")
      .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))
     )

26/05/28 18:52:42 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [15]:
# spark = SparkSession.builder.appName("Jupyter").getOrCreate()

# events = spark.read.option("header", "true").csv("/home/iceberg/data/events.csv").withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))
# devices = spark.read.option("header","true").csv("/home/iceberg/data/devices.csv")

# df = events.join(devices,on="device_id",how="left")
# df = df.withColumnsRenamed({'browser_type': 'browser_family', 'os_type': 'os_family'})

# df.show()

In [3]:
sorted = df.repartition(10, col("event_date"))\
    .withColumn("partition_id", spark_partition_id())\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.show()
sortedTwo.show()

+-----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+------------+
|    user_id|  device_id|            referrer|                host|                 url|          event_time|         event_date|partition_id|
+-----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+------------+
| 1129583063|  532630305|                NULL|admin.zachwilson....|                   /|2021-01-07 09:21:...|2021-01-07 00:00:00|           0|
| -648945006| 1088283544|                NULL|    www.eczachly.com|                   /|2021-01-07 02:58:...|2021-01-07 00:00:00|           0|
|-1871780024| -158310583|                NULL|    www.eczachly.com|                   /|2021-01-07 04:17:...|2021-01-07 00:00:00|           0|
|  203689086| 1088283544|                NULL|    www.eczachly.com|/blog/what-exactl...|2021-01-07 10:03:...|2021-01-07 00:00:00|           0|

[Stage 4:=======>                                                   (1 + 7) / 8]

+-----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+
|    user_id|  device_id|            referrer|                host|                 url|          event_time|         event_date|
+-----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+
| 1272828233| -643696601|                NULL|admin.zachwilson....|                   /|2021-01-02 13:53:...|2021-01-02 00:00:00|
|  747494706|  532630305|                NULL|admin.zachwilson....|                   /|2021-01-02 19:36:...|2021-01-02 00:00:00|
| 2110046626|  898871897|                NULL|admin.zachwilson....|       /wp-login.php|2021-01-02 19:57:...|2021-01-02 00:00:00|
| 1272828233| -643696601|                NULL|admin.zachwilson....|                   /|2021-01-02 21:05:...|2021-01-02 00:00:00|
| 1272828233| -643696601|                NULL|admin.zachwilson....|                   /|20

In [4]:
# .sortWithinPartitions() sorts within partitions, whereas .sort() is a global sort, which is very slow

# Note - exchange is synonymous with Shuffle

In [5]:
# df_bnr = spark.read.option("header", "true").csv("/home/iceberg/data/events.csv").withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))

In [6]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()
sortedTwo.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, cast(event_time#22 as timestamp) AS event_time#138, event_date#29]
   +- Sort [event_date#29 ASC NULLS FIRST, host#20 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#29, 10), REPARTITION_BY_NUM, [plan_id=129]
         +- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, event_time#22, date_trunc(day, cast(event_time#22 as timestamp), Some(Etc/UTC)) AS event_date#29]
            +- FileScan csv [user_id#17,device_id#18,referrer#19,host#20,url#21,event_time#22] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/events.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<user_id:string,device_id:string,referrer:string,host:string,url:string,event_time:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referr

In [7]:
# sortWithinPartitions do not require a shuffle,
# while the global sort, naturally, does

In [8]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

++
||
++
++

In [9]:
%%sql

DROP TABLE IF EXISTS bootcamp.events

++
||
++
++

In [10]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_sorted

++
||
++
++

In [11]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_unsorted

++
||
++
++

In [12]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));


++
||
++
++

In [13]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));

++
||
++
++

In [14]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (year(event_date));

++
||
++
++

In [15]:
start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [16]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files

size,num_files,sorted
4958022,4,sorted
5053371,4,unsorted


In [ ]:
# the we were trying to show here is that we do get some disk usage saving
# because of compression from the run length encoding,
# this compression is the most effective when we have the data sorted by its fields with lowest cardinalities